# Step 1 — Cleaning the ECB Survey of Professional Forecasters (SPF) Data

## Purpose

This notebook reads the raw ECB SPF individual-round CSV files, extracts the
**point forecasts** for **HICP inflation** and **real GDP growth**, and
produces a single clean panel dataset.

The final table has one row per *forecaster × survey round × target period*
and contains the columns we need for the later modelling steps:

| column | meaning |
|---|---|
| `survey_round` | the quarter the survey was conducted (e.g. `2020Q1`) |
| `variable` | `HICP` (inflation) or `RGDP` (real GDP growth) |
| `target_period` | the period the forecast refers to (e.g. `2020`, `2021Dec`, `2020Q3`) |
| `fct_source` | anonymous forecaster ID (integer, stable across rounds) |
| `point_forecast` | the forecaster's point estimate |
| `horizon_type` | label for the forecast horizon: `current_year`, `next_year`, `year_after_next`, `longer_term`, `rolling_1y`, `rolling_2y` |

We keep **all** available calendar-year and rolling horizons at this stage.
Downstream notebooks will filter to the rolling one-year-ahead horizon.

---
## 1 — Imports and setup

In [1]:
import os
import glob
import re

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 20)
pd.set_option("display.max_rows", 30)
pd.set_option("display.width", 120)

print(f"pandas  {pd.__version__}")
print(f"numpy   {np.__version__}")

pandas  3.0.2
numpy   2.4.4


In [2]:
# Path to the folder that contains one CSV per survey round (1999Q1.csv … 2026Q2.csv)
DATA_DIR = os.path.join("Data", "SPF_individual_forecasts")

csv_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.csv")))
print(f"Found {len(csv_files)} survey-round files.")
print(f"First : {os.path.basename(csv_files[0])}")
print(f"Last  : {os.path.basename(csv_files[-1])}")

Found 110 survey-round files.
First : 1999Q1.csv
Last  : 2026Q2.csv


---
## 2 — Understanding the raw CSV layout

Each CSV file corresponds to **one survey round** (e.g. `2020Q1.csv`).
Inside a single file, several tables are stacked vertically, separated by
blank rows:

1. **HICP inflation** — header row starts with
   `INFLATION EXPECTATIONS; YEAR-ON-YEAR CHANGE IN HICP`
2. **Core inflation (HICPX)** — header starts with
   `CORE INFLATION EXPECTATIONS; YEAR-ON-YEAR CHANGE IN CORE`
3. **Real GDP growth** — header starts with
   `GROWTH EXPECTATIONS; YEAR-ON-YEAR CHANGE IN REAL GDP`
4. **Unemployment** — header starts with
   `EXPECTED UNEMPLOYMENT RATE`

Below the four forecast tables there may also be an assumptions block
(ECB interest rate, oil prices, exchange rates, labour costs).

### Column structure inside each table

| column | content |
|---|---|
| `TARGET_PERIOD` | The period being forecast. Format depends on the variable and horizon: a plain year like `2020` for calendar-year forecasts, a month like `2020Dec` for HICP rolling horizons, or a quarter like `2020Q3` for GDP rolling horizons. |
| `FCT_SOURCE` | Integer ID of the individual forecaster (anonymous, but stable over time). |
| `POINT` | The forecaster's point estimate (single number). |
| remaining columns | Probability bins for the density forecast (we ignore these). |

We only need `TARGET_PERIOD`, `FCT_SOURCE`, and `POINT`.

### Forecast horizons

The ECB asks forecasters for **up to six horizons** (since 2001 Q2):

1. **Current calendar year** — e.g. in 2020Q1 the target period `2020`.
2. **Next calendar year** — e.g. `2021`.
3. **Calendar year after next** — e.g. `2022`.
4. **Longer-term** — four or five calendar years ahead.
5. **Rolling 1-year-ahead** — for HICP this is a month
   (e.g. `2020Dec`), for GDP a quarter (e.g. `2020Q3`). The idea is
   that the target is always approximately 12 months from the last
   available data release.
6. **Rolling 2-year-ahead** — same concept, two years out
   (e.g. `2021Dec` / `2021Q3`).

The rolling horizons are the most interesting for our project because
they give a **fixed forecasting distance**, making comparisons across
time cleaner.

Let's peek at a raw file to verify our understanding:

In [3]:
# Quick peek at the first 10 lines of 2020Q1.csv
sample_path = os.path.join(DATA_DIR, "2020Q1.csv")
with open(sample_path, "r") as f:
    for i, line in enumerate(f):
        if i < 10:
            print(f"line {i+1:>4}: {line.rstrip()[:120]}")

line    1: INFLATION EXPECTATIONS; YEAR-ON-YEAR CHANGE IN HICP,,,,,,,,,,,,,,,,,,,,,,,
line    2: TARGET_PERIOD,FCT_SOURCE,POINT,TN1_0,FN1_0TN0_6,FN0_5TN0_1,F0_0T0_4,F0_5T0_9,F1_0T1_4,F1_5T1_9,F2_0T2_4,F2_5T2_9,F3_0T3_
line    3: 2020,1,1.2,,1,2,4,15,44,22,8,3,1,,,,,,,,,,,
line    4: 2020,2,1.4,0,0,0,0,0,50,50,0,0,0,0,0,,,,,,,,,
line    5: 2020,4,1.2,,,,10,20,40,20,10,,,,,,,,,,,,,
line    6: 2020,5,1,,,,7.5,25,40,20,7.5,,,,,,,,,,,,,
line    7: 2020,6,1.2,,,,,10,50,25,10,5,,,,,,,,,,,,
line    8: 2020,7,1.1766426795,,,,,,,,,,,,,,,,,,,,,
line    9: 2020,8,1,,,,5,30,35,20,10,,,,,,,,,,,,,
line   10: 2020,10,1,,,,5,15,70,10,,,,,,,,,,,,,,


Line 1 is the section header for HICP. Line 2 is the column header row.
Lines 3 onwards contain the actual data. Note how `TARGET_PERIOD` values
like `2020` (calendar year) and `2020Dec` (rolling 1-year-ahead) are
intermixed — the section header tells us the *variable* (HICP vs RGDP),
and the target period format tells us the *horizon type*.

---
## 3 — Parsing a single CSV file

Our strategy for parsing one file:

1. Read the file line by line.
2. Detect section headers to know which variable (HICP / RGDP) we are in.
3. When we see a `TARGET_PERIOD` header row, read subsequent data rows
   until we hit a blank line or a new section header.
4. Keep only `TARGET_PERIOD`, `FCT_SOURCE`, and `POINT`.
5. Tag every row with the current variable name.

We wrap this in a function so we can apply it to all 110 files.

In [4]:
# -------------------------------------------------------------------
# Section-header patterns that tell us which variable block we are in.
# We only care about HICP and RGDP; the function skips everything else.
# -------------------------------------------------------------------
SECTION_PATTERNS = {
    "HICP": re.compile(r"^INFLATION EXPECTATIONS;\s*YEAR-ON-YEAR CHANGE IN HICP"),
    "RGDP": re.compile(r"^GROWTH EXPECTATIONS;\s*YEAR-ON-YEAR CHANGE IN REAL GDP"),
}

# Patterns that signal the START of a section we want to skip.
# When we see one of these while inside an HICP/RGDP block we stop reading.
SKIP_PATTERNS = [
    re.compile(r"^CORE INFLATION"),
    re.compile(r"^EXPECTED UNEMPLOYMENT"),
    re.compile(r"^ASSUMPTIONS"),
]


def parse_spf_file(filepath: str) -> pd.DataFrame:
    """
    Parse one SPF survey-round CSV file and return a DataFrame with columns:
        TARGET_PERIOD, FCT_SOURCE, POINT, variable

    Only HICP and RGDP blocks are extracted; all other blocks are skipped.
    """
    rows = []  # accumulator for (target_period, fct_source, point, variable)
    current_var = None  # which variable block are we inside?

    with open(filepath, "r") as f:
        for raw_line in f:
            line = raw_line.strip()

            # --- Check for section headers we want ---
            matched_var = None
            for var_name, pattern in SECTION_PATTERNS.items():
                if pattern.search(line):
                    matched_var = var_name
                    break

            if matched_var:
                current_var = matched_var
                continue  # move to the next line (column header follows)

            # --- Check for section headers we want to skip ---
            if any(p.search(line) for p in SKIP_PATTERNS):
                current_var = None  # we are no longer in an HICP/RGDP block
                continue

            # --- Skip column-header rows and blank lines ---
            if line.startswith("TARGET_PERIOD") or line == "" or line.replace(",", "") == "":
                continue

            # --- If we are inside an HICP or RGDP block, parse the data row ---
            if current_var is not None:
                fields = line.split(",")
                target_period = fields[0].strip()
                fct_source = fields[1].strip()
                point_raw = fields[2].strip()

                # Skip rows where POINT is missing (some forecasters only
                # provide density forecasts without a point estimate)
                if point_raw == "" or target_period == "" or fct_source == "":
                    continue

                rows.append((target_period, fct_source, point_raw, current_var))

    df = pd.DataFrame(rows, columns=["TARGET_PERIOD", "FCT_SOURCE", "POINT", "variable"])
    return df


# Quick test on 2020Q1
test_df = parse_spf_file(os.path.join(DATA_DIR, "2020Q1.csv"))
print(f"Rows parsed from 2020Q1.csv: {len(test_df)}")
print(f"Variables found: {test_df['variable'].unique()}")
print(f"Target periods:  {sorted(test_df['TARGET_PERIOD'].unique())}")
test_df.head(8)

Rows parsed from 2020Q1.csv: 723
Variables found: <StringArray>
['HICP', 'RGDP']
Length: 2, dtype: str
Target periods:  ['2020', '2020Dec', '2020Q3', '2021', '2021Dec', '2021Q3', '2022', '2024']


,TARGET_PERIOD,FCT_SOURCE,POINT,variable
0,2020,1,1.2,HICP
1,2020,2,1.4,HICP
2,2020,4,1.2,HICP
3,2020,5,1,HICP
4,2020,6,1.2,HICP
5,2020,7,1.1766426795,HICP
6,2020,8,1,HICP
7,2020,10,1,HICP


The parser correctly picks up only the HICP and RGDP sections.
For HICP the rolling horizons are months (e.g. `2020Dec`, `2021Dec`)
and for RGDP they are quarters (e.g. `2020Q3`, `2021Q3`).

---
## 4 — Parsing all 110 survey rounds

We loop over every CSV in the `Data/SPF_individual_forecasts` folder.
The **survey round** (e.g. `2020Q1`) is extracted from the filename.

In [5]:
frames = []

for fpath in csv_files:
    # Extract the survey round from the filename: "2020Q1.csv" → "2020Q1"
    survey_round = os.path.splitext(os.path.basename(fpath))[0]

    df = parse_spf_file(fpath)
    df["survey_round"] = survey_round  # tag every row
    frames.append(df)

raw = pd.concat(frames, ignore_index=True)
print(f"Total rows across all rounds: {len(raw):,}")
print(f"Survey rounds: {raw['survey_round'].nunique()}")
raw.head()

Total rows across all rounds: 61,677
Survey rounds: 110


,TARGET_PERIOD,FCT_SOURCE,POINT,variable,survey_round
0,1999,1,1,HICP,1999Q1
1,1999,2,1,HICP,1999Q1
2,1999,3,.8,HICP,1999Q1
3,1999,4,1.2,HICP,1999Q1
4,1999,5,.8,HICP,1999Q1


---
## 5 — Type conversions and basic cleaning

The values we parsed are still strings. We need to:

1. Convert `FCT_SOURCE` to integer (forecaster IDs are integers).
2. Convert `POINT` to float (the point forecast).
3. Drop any row where the conversion fails — this catches the rare
   cases where a field is garbled or empty.

In [6]:
# Convert POINT to numeric — non-numeric strings become NaN
raw["POINT"] = pd.to_numeric(raw["POINT"], errors="coerce")

# Convert FCT_SOURCE to integer
raw["FCT_SOURCE"] = pd.to_numeric(raw["FCT_SOURCE"], errors="coerce")

# Count how many rows have a missing point forecast after conversion
n_missing_point = raw["POINT"].isna().sum()
n_missing_fct = raw["FCT_SOURCE"].isna().sum()
print(f"Missing POINT values after conversion: {n_missing_point}")
print(f"Missing FCT_SOURCE after conversion:   {n_missing_fct}")

# Drop rows with missing point forecasts
raw = raw.dropna(subset=["POINT", "FCT_SOURCE"]).copy()
raw["FCT_SOURCE"] = raw["FCT_SOURCE"].astype(int)

print(f"\nRows remaining: {len(raw):,}")
raw.dtypes

Missing POINT values after conversion: 0
Missing FCT_SOURCE after conversion:   0

Rows remaining: 61,677


TARGET_PERIOD        str
FCT_SOURCE         int64
POINT            float64
variable             str
survey_round         str
dtype: object

---
## 6 — Classifying the forecast horizon

This is the most important cleaning step. Each row has a `TARGET_PERIOD`
and a `survey_round`. From these two we can figure out **which forecast
horizon** the row belongs to.

### Horizon classification rules

| horizon label | how to detect it |
|---|---|
| `rolling_1y` | HICP: target is a month string (e.g. `2020Dec`); RGDP: target is a quarter string (e.g. `2020Q3`). The target year equals `survey_year` or `survey_year + 1` — it's roughly 1 year from the survey date. |
| `rolling_2y` | Same format as rolling_1y but the target year is roughly 2 years out (target year ≈ survey_year + 1 or + 2). |
| `current_year` | Target is a plain year equal to the survey year. |
| `next_year` | Target is a plain year equal to survey year + 1. |
| `year_after_next` | Target is a plain year equal to survey year + 2. |
| `longer_term` | Target is a plain year ≥ survey year + 3. |

The trick is that **rolling horizons always contain a month or quarter
suffix** (like `Dec` or `Q3`), whereas calendar-year horizons are just
a four-digit year.

We implement this classification with a function:

In [7]:
# Regex patterns for target-period formats
RE_YEAR_ONLY = re.compile(r"^(\d{4})$")                     # e.g. "2020"
RE_MONTH = re.compile(r"^(\d{4})(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)$")  # e.g. "2020Dec"
RE_QUARTER = re.compile(r"^(\d{4})Q([1-4])$")               # e.g. "2020Q3"


def classify_horizon(target_period: str, survey_round: str) -> str:
    """
    Given a TARGET_PERIOD string and the survey_round (e.g. '2020Q1'),
    return a horizon label: 'rolling_1y', 'rolling_2y', 'current_year',
    'next_year', 'year_after_next', or 'longer_term'.
    """
    survey_year = int(survey_round[:4])

    # --- Rolling horizons (contain a month or quarter suffix) ---
    m_month = RE_MONTH.match(target_period)
    m_quarter = RE_QUARTER.match(target_period)

    if m_month or m_quarter:
        target_year = int(m_month.group(1)) if m_month else int(m_quarter.group(1))
        diff = target_year - survey_year
        if diff <= 1:
            return "rolling_1y"
        elif diff <= 2:
            return "rolling_2y"
        else:
            return "rolling_longer"

    # --- Calendar-year horizons (plain four-digit year) ---
    m_year = RE_YEAR_ONLY.match(target_period)
    if m_year:
        target_year = int(m_year.group(1))
        diff = target_year - survey_year
        if diff == 0:
            return "current_year"
        elif diff == 1:
            return "next_year"
        elif diff == 2:
            return "year_after_next"
        else:
            return "longer_term"

    return "unknown"


# Apply the classification to every row
raw["horizon_type"] = raw.apply(
    lambda r: classify_horizon(r["TARGET_PERIOD"], r["survey_round"]),
    axis=1,
)

print("Horizon type distribution:")
print(raw["horizon_type"].value_counts().sort_index())

Horizon type distribution:
horizon_type
current_year       12490
longer_term         9115
next_year          12236
rolling_1y         14748
rolling_2y          5489
rolling_longer       253
year_after_next     7346
Name: count, dtype: int64


In [8]:
# Sanity check: there should be zero "unknown" horizons
n_unknown = (raw["horizon_type"] == "unknown").sum()
if n_unknown > 0:
    print(f"WARNING: {n_unknown} rows with unknown horizon type!")
    print(raw.loc[raw["horizon_type"] == "unknown", "TARGET_PERIOD"].unique())
else:
    print("All rows classified successfully — no unknowns.")

All rows classified successfully — no unknowns.


---
## 7 — Rename columns to final schema

We rename the raw column names to the clean, lowercase names that we
will use throughout the project.

In [9]:
clean = raw.rename(columns={
    "TARGET_PERIOD": "target_period",
    "FCT_SOURCE":    "fct_source",
    "POINT":         "point_forecast",
})

# Reorder columns into a logical sequence
clean = clean[["survey_round", "variable", "target_period",
               "fct_source", "point_forecast", "horizon_type"]]

clean.head(10)

,survey_round,variable,target_period,fct_source,point_forecast,horizon_type
0,1999Q1,HICP,1999,1,1.0,current_year
1,1999Q1,HICP,1999,2,1.0,current_year
2,1999Q1,HICP,1999,3,0.8,current_year
3,1999Q1,HICP,1999,4,1.2,current_year
4,1999Q1,HICP,1999,5,0.8,current_year
5,1999Q1,HICP,1999,6,0.9,current_year
6,1999Q1,HICP,1999,7,0.8,current_year
7,1999Q1,HICP,1999,9,0.7,current_year
8,1999Q1,HICP,1999,10,1.2,current_year
9,1999Q1,HICP,1999,11,1.2,current_year


---
## 8 — Extract the rolling 1-year-ahead and next-year forecast columns

The user asked for two specific forecast columns:

- **`rolling_1y_forecast`** — the point forecast for the rolling
  one-year-ahead horizon.
- **`next_year_forecast`** — the point forecast for the next calendar year.

We pivot the data so that each forecaster × survey round has these
two numbers side by side. This is useful because we can later compare
the two horizons and check consistency.

### Why keep both?

The **rolling 1-year-ahead** horizon is the main one for the neural
network project (it gives a fixed forecast distance). The **next calendar
year** horizon is the most commonly reported in the SPF and is useful as
an additional feature or cross-check.

In [10]:
# --- Rolling 1-year-ahead forecasts ---
rolling_1y = (
    clean
    .loc[clean["horizon_type"] == "rolling_1y"]
    .rename(columns={"point_forecast": "rolling_1y_forecast",
                     "target_period":  "rolling_1y_target"})
    [["survey_round", "variable", "fct_source",
      "rolling_1y_target", "rolling_1y_forecast"]]
)

# --- Next-year calendar-year forecasts ---
next_year = (
    clean
    .loc[clean["horizon_type"] == "next_year"]
    .rename(columns={"point_forecast": "next_year_forecast",
                     "target_period":  "next_year_target"})
    [["survey_round", "variable", "fct_source",
      "next_year_target", "next_year_forecast"]]
)

print(f"Rolling 1y rows : {len(rolling_1y):,}")
print(f"Next year rows  : {len(next_year):,}")

Rolling 1y rows : 14,748
Next year rows  : 12,236


In [11]:
# Merge the two horizon slices together
# We use an outer merge so we keep forecasters who reported one but not the other
merged = pd.merge(
    rolling_1y,
    next_year,
    on=["survey_round", "variable", "fct_source"],
    how="outer",
)

print(f"Merged rows: {len(merged):,}")
print(f"\nMissing rolling_1y_forecast: {merged['rolling_1y_forecast'].isna().sum():,}")
print(f"Missing next_year_forecast:  {merged['next_year_forecast'].isna().sum():,}")
merged.head(10)

Merged rows: 16,374

Missing rolling_1y_forecast: 1,626
Missing next_year_forecast:  123


,survey_round,variable,fct_source,rolling_1y_target,rolling_1y_forecast,next_year_target,next_year_forecast
0,1999Q1,HICP,1,1999Dec,1.2,2000,1.5
1,1999Q1,HICP,1,2000Dec,1.7,2000,1.5
2,1999Q1,HICP,2,1999Dec,1.2,2000,1.2
3,1999Q1,HICP,2,2000Dec,1.2,2000,1.2
4,1999Q1,HICP,3,1999Dec,0.8,2000,1.0
5,1999Q1,HICP,3,2000Dec,1.2,2000,1.0
6,1999Q1,HICP,4,1999Dec,1.3,2000,1.6
7,1999Q1,HICP,4,2000Dec,1.8,2000,1.6
8,1999Q1,HICP,5,1999Dec,1.1,2000,1.4
9,1999Q1,HICP,5,2000Dec,1.5,2000,1.4


---
## 9 — Add current-year forecast as an extra feature

The current-year forecast is another useful piece of information.
By the time Q3 or Q4 rolls around, the current-year forecast is nearly
a realized value, so it captures how much information about the
macro economy the forecaster already has. We add it as an additional
column.

In [12]:
current_year = (
    clean
    .loc[clean["horizon_type"] == "current_year"]
    .rename(columns={"point_forecast": "current_year_forecast",
                     "target_period":  "current_year_target"})
    [["survey_round", "variable", "fct_source",
      "current_year_target", "current_year_forecast"]]
)

merged = pd.merge(
    merged,
    current_year,
    on=["survey_round", "variable", "fct_source"],
    how="left",
)

print(f"After adding current-year forecast: {len(merged):,} rows")
merged.head(10)

After adding current-year forecast: 16,374 rows


,survey_round,variable,fct_source,rolling_1y_target,rolling_1y_forecast,next_year_target,next_year_forecast,current_year_target,current_year_forecast
0,1999Q1,HICP,1,1999Dec,1.2,2000,1.5,1999,1.0
1,1999Q1,HICP,1,2000Dec,1.7,2000,1.5,1999,1.0
2,1999Q1,HICP,2,1999Dec,1.2,2000,1.2,1999,1.0
3,1999Q1,HICP,2,2000Dec,1.2,2000,1.2,1999,1.0
4,1999Q1,HICP,3,1999Dec,0.8,2000,1.0,1999,0.8
5,1999Q1,HICP,3,2000Dec,1.2,2000,1.0,1999,0.8
6,1999Q1,HICP,4,1999Dec,1.3,2000,1.6,1999,1.2
7,1999Q1,HICP,4,2000Dec,1.8,2000,1.6,1999,1.2
8,1999Q1,HICP,5,1999Dec,1.1,2000,1.4,1999,0.8
9,1999Q1,HICP,5,2000Dec,1.5,2000,1.4,1999,0.8


---
## 10 — Parse the survey round into year and quarter columns

Having `survey_year` and `survey_quarter` as separate numeric columns
will be handy for sorting, plotting, and later for creating time-based
features like "survey quarter" dummies.

In [13]:
merged["survey_year"] = merged["survey_round"].str[:4].astype(int)
merged["survey_quarter"] = merged["survey_round"].str[-1].astype(int)

merged.head()

,survey_round,variable,fct_source,rolling_1y_target,rolling_1y_forecast,next_year_target,next_year_forecast,current_year_target,current_year_forecast,survey_year,survey_quarter
0,1999Q1,HICP,1,1999Dec,1.2,2000,1.5,1999,1.0,1999,1
1,1999Q1,HICP,1,2000Dec,1.7,2000,1.5,1999,1.0,1999,1
2,1999Q1,HICP,2,1999Dec,1.2,2000,1.2,1999,1.0,1999,1
3,1999Q1,HICP,2,2000Dec,1.2,2000,1.2,1999,1.0,1999,1
4,1999Q1,HICP,3,1999Dec,0.8,2000,1.0,1999,0.8,1999,1


---
## 11 — Summary statistics and quality checks

In [14]:
print("=" * 60)
print("DATASET OVERVIEW")
print("=" * 60)
print(f"Total rows             : {len(merged):,}")
print(f"Unique survey rounds   : {merged['survey_round'].nunique()}")
print(f"Unique forecasters     : {merged['fct_source'].nunique()}")
print(f"Variables              : {sorted(merged['variable'].unique())}")
print(f"Year range             : {merged['survey_year'].min()} – {merged['survey_year'].max()}")
print()

for var in ["HICP", "RGDP"]:
    sub = merged[merged["variable"] == var]
    print(f"--- {var} ---")
    print(f"  Rows                   : {len(sub):,}")
    print(f"  Survey rounds          : {sub['survey_round'].nunique()}")
    print(f"  Forecasters            : {sub['fct_source'].nunique()}")
    print(f"  rolling_1y available    : {sub['rolling_1y_forecast'].notna().sum():,}")
    print(f"  next_year available     : {sub['next_year_forecast'].notna().sum():,}")
    print(f"  current_year available  : {sub['current_year_forecast'].notna().sum():,}")
    print()

DATASET OVERVIEW
Total rows             : 16,374
Unique survey rounds   : 110
Unique forecasters     : 113
Variables              : ['HICP', 'RGDP']
Year range             : 1999 – 2026

--- HICP ---
  Rows                   : 7,528
  Survey rounds          : 110
  Forecasters            : 113
  rolling_1y available    : 6,726
  next_year available     : 7,472
  current_year available  : 7,525

--- RGDP ---
  Rows                   : 8,846
  Survey rounds          : 110
  Forecasters            : 113
  rolling_1y available    : 8,022
  next_year available     : 8,779
  current_year available  : 8,844



In [15]:
# Descriptive statistics for the point forecasts
merged[["rolling_1y_forecast", "next_year_forecast", "current_year_forecast"]].describe().round(2)

,rolling_1y_forecast,next_year_forecast,current_year_forecast
count,14748.00,16251.00,16369.00
mean,1.70,1.82,1.57
std,1.12,0.87,1.71
min,-10.95,-2.01,-13.00
25%,1.30,1.40,1.04
50%,1.70,1.70,1.70
75%,2.00,2.10,2.20
max,16.60,14.50,9.30


In [16]:
# Check for obvious outliers: point forecasts outside a plausible range
# HICP and RGDP in the euro area have historically stayed within roughly -15% to +15%
for var in ["HICP", "RGDP"]:
    sub = merged[merged["variable"] == var]
    for col in ["rolling_1y_forecast", "next_year_forecast", "current_year_forecast"]:
        vals = sub[col].dropna()
        extreme = vals[(vals < -15) | (vals > 15)]
        if len(extreme) > 0:
            print(f"  {var} / {col}: {len(extreme)} values outside [-15, 15]")
        else:
            print(f"  {var} / {col}: all values in plausible range")

  HICP / rolling_1y_forecast: all values in plausible range
  HICP / next_year_forecast: all values in plausible range
  HICP / current_year_forecast: all values in plausible range
  RGDP / rolling_1y_forecast: 1 values outside [-15, 15]
  RGDP / next_year_forecast: all values in plausible range
  RGDP / current_year_forecast: all values in plausible range


---
## 12 — Inspect sample rows to make sure everything looks right

In [17]:
# Show a few rows for HICP, 2020Q1
print("HICP forecasts in 2020Q1 (first 8 forecasters):")
merged.loc[
    (merged["survey_round"] == "2020Q1") & (merged["variable"] == "HICP")
].head(8)

HICP forecasts in 2020Q1 (first 8 forecasters):


,survey_round,variable,fct_source,rolling_1y_target,rolling_1y_forecast,next_year_target,next_year_forecast,current_year_target,current_year_forecast,survey_year,survey_quarter
12409,2020Q1,HICP,1,2020Dec,1.2,2021,1.5,2020,1.2,2020,1
12410,2020Q1,HICP,1,2021Dec,1.5,2021,1.5,2020,1.2,2020,1
12411,2020Q1,HICP,2,2020Dec,1.4,2021,1.5,2020,1.4,2020,1
12412,2020Q1,HICP,2,2021Dec,1.5,2021,1.5,2020,1.4,2020,1
12413,2020Q1,HICP,4,2020Dec,1.2,2021,1.2,2020,1.2,2020,1
12414,2020Q1,HICP,4,2021Dec,1.2,2021,1.2,2020,1.2,2020,1
12415,2020Q1,HICP,5,2020Dec,1.0,2021,1.5,2020,1.0,2020,1
12416,2020Q1,HICP,5,2021Dec,1.5,2021,1.5,2020,1.0,2020,1


In [18]:
# Show a few rows for RGDP, 2020Q1
print("RGDP forecasts in 2020Q1 (first 8 forecasters):")
merged.loc[
    (merged["survey_round"] == "2020Q1") & (merged["variable"] == "RGDP")
].head(8)

RGDP forecasts in 2020Q1 (first 8 forecasters):


,survey_round,variable,fct_source,rolling_1y_target,rolling_1y_forecast,next_year_target,next_year_forecast,current_year_target,current_year_forecast,survey_year,survey_quarter
12533,2020Q1,RGDP,1,2020Q3,1.0,2021,1.3,2020,0.9,2020,1
12534,2020Q1,RGDP,1,2021Q3,1.3,2021,1.3,2020,0.9,2020,1
12535,2020Q1,RGDP,2,2020Q3,1.4,2021,1.7,2020,1.4,2020,1
12536,2020Q1,RGDP,2,2021Q3,1.7,2021,1.7,2020,1.4,2020,1
12537,2020Q1,RGDP,4,2020Q3,0.6,2021,1.1,2020,0.6,2020,1
12538,2020Q1,RGDP,4,2021Q3,1.2,2021,1.1,2020,0.6,2020,1
12539,2020Q1,RGDP,5,2020Q3,1.0,2021,1.5,2020,1.0,2020,1
12540,2020Q1,RGDP,5,2021Q3,1.5,2021,1.5,2020,1.0,2020,1


---
## 13 — Also keep the long-form (all horizons) dataset

The `merged` table above is the **wide** format with rolling_1y,
next_year, and current_year as separate columns. But it's also useful
to keep the **long** format (`clean`) that has every single horizon as
a separate row, in case we need to look at year-after-next or longer-term
forecasts later.

We save both.

In [19]:
# Add survey_year and survey_quarter to the long-form dataset too
clean["survey_year"] = clean["survey_round"].str[:4].astype(int)
clean["survey_quarter"] = clean["survey_round"].str[-1].astype(int)

print("Long-form dataset shape:", clean.shape)
print("Wide-form dataset shape:", merged.shape)

Long-form dataset shape: (61677, 8)
Wide-form dataset shape: (16374, 11)


---
## 14 — Save the cleaned datasets to CSV

In [20]:
# Create output directory if it doesn't exist
os.makedirs("Data", exist_ok=True)

# Long form: one row per forecaster × round × target period × variable
clean.to_csv(os.path.join("Data", "spf_clean_long.csv"), index=False)
print(f"Saved long-form dataset : Data/spf_clean_long.csv  ({len(clean):,} rows)")

# Wide form: one row per forecaster × round × variable, with rolling_1y / next_year / current_year columns
merged.to_csv(os.path.join("Data", "spf_clean_wide.csv"), index=False)
print(f"Saved wide-form dataset : Data/spf_clean_wide.csv  ({len(merged):,} rows)")

Saved long-form dataset : Data/spf_clean_long.csv  (61,677 rows)
Saved wide-form dataset : Data/spf_clean_wide.csv  (16,374 rows)


---
## 15 — Final schema of the wide dataset

This is the dataset we will use going forward. Here is every column
and what it means:

In [21]:
schema_info = {
    "column": merged.columns.tolist(),
    "dtype": [str(merged[c].dtype) for c in merged.columns],
    "non_null": [merged[c].notna().sum() for c in merged.columns],
    "example": [merged[c].dropna().iloc[0] if merged[c].notna().any() else None for c in merged.columns],
}
pd.DataFrame(schema_info)

,column,dtype,non_null,example
0,survey_round,str,16374,1999Q1
1,variable,str,16374,HICP
2,fct_source,int64,16374,1
3,rolling_1y_target,str,14748,1999Dec
4,rolling_1y_forecast,float64,14748,1.2
5,next_year_target,str,16251,2000
6,next_year_forecast,float64,16251,1.5
7,current_year_target,str,16369,1999
8,current_year_forecast,float64,16369,1.0
9,survey_year,int64,16374,1999


### Column descriptions

| column | description |
|---|---|
| `survey_round` | Quarter when the ECB conducted the survey (e.g. `2020Q1`). Extracted from the filename. |
| `variable` | Which macroeconomic variable: `HICP` = year-on-year change in the Harmonised Index of Consumer Prices (inflation); `RGDP` = year-on-year change in real GDP (growth). |
| `fct_source` | Anonymous integer ID for each forecaster. The same number refers to the same institution across all rounds. |
| `rolling_1y_target` | The specific target period for the rolling one-year-ahead forecast (e.g. `2020Dec` for HICP, `2020Q3` for RGDP). |
| `rolling_1y_forecast` | The forecaster's point estimate for the rolling one-year-ahead horizon. This is our **main forecast** column. |
| `next_year_target` | The calendar year being forecast as "next year" (e.g. `2021` when surveyed in 2020). |
| `next_year_forecast` | The forecaster's point estimate for the next calendar year. |
| `current_year_target` | The calendar year being forecast as "current year". |
| `current_year_forecast` | The forecaster's point estimate for the current calendar year. |
| `survey_year` | Numeric year of the survey (integer). |
| `survey_quarter` | Numeric quarter of the survey (1–4). |

---
## Summary

We started with 110 raw CSV files from the ECB SPF, each containing
multiple stacked tables for different macro variables.

**What we did:**

1. Wrote a parser that reads each CSV, identifies the HICP and RGDP
   sections by their header text, and extracts `TARGET_PERIOD`,
   `FCT_SOURCE`, and `POINT`.
2. Tagged every row with the `survey_round` from the filename.
3. Converted types (string → numeric) and dropped rows with missing
   point forecasts.
4. Classified every row into a horizon type (`rolling_1y`, `next_year`,
   `current_year`, etc.) based on the target-period format and its
   distance from the survey year.
5. Pivoted the three most relevant horizons into separate columns:
   `rolling_1y_forecast`, `next_year_forecast`, `current_year_forecast`.
6. Added `survey_year` and `survey_quarter` for convenience.
7. Saved both a long-form and a wide-form CSV.

**What we have now:**

- A clean panel dataset at the *forecaster × survey round × variable*
  level.
- Point forecasts for three horizons, ready for the next step:
  merging with realized outcomes and computing forecast errors.